# 02 - PDF Extraction

**Goal:** Extract clean text and tables from all 70 PDF files using PyMuPDF.

**Input:** `data/raw/*.pdf` + `data/processed/inventory.json`  
**Output:** `data/processed/pdf_extracted/` - one `.json` per PDF with:
- Clean full text (page by page)
- Extracted tables (as markdown strings, ready for chunking)
- Metadata: source, pages, section headings detected

**Two extraction paths:**
1. **Standard PDFs** → PyMuPDF `get_text()` per page
2. **PDFs with tables** (flagged in inventory) → PyMuPDF `find_tables()` for table content + surrounding text

**Key decisions:**
- We preserve page boundaries - useful for metadata
- Tables are converted to markdown format (`| col | col |`) - LLM handles markdown well
- We detect section headings via font size heuristics - used for chunk context later
- Output is clean, normalized text - no ligatures, no weird spacing

## Setup

In [1]:
import json
import re
from pathlib import Path
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

import fitz
from tqdm import tqdm

# Paths
NOTEBOOK_DIR = Path().resolve()
BACKEND_DIR = NOTEBOOK_DIR.parent
RAW_DIR = BACKEND_DIR / 'data' / 'raw'
PROCESSED_DIR = BACKEND_DIR / 'data' / 'processed'
PDF_OUT_DIR = PROCESSED_DIR / 'pdf_extracted'
PDF_OUT_DIR.mkdir(parents=True, exist_ok=True)

INVENTORY_PATH = PROCESSED_DIR / 'inventory.json'

print('Setup complete')
print(f'  Output dir: {PDF_OUT_DIR}')

Setup complete
  Output dir: D:\Cybernetic Gym Assistant\backend\data\processed\pdf_extracted


## 1. Load Inventory - PDF files only

In [2]:
with open(INVENTORY_PATH, 'r', encoding='utf-8') as f:
    inventory = json.load(f)

pdf_files = [
    fi for fi in inventory['files']
    if fi['file_type'] == 'pdf' and fi.get('destination') != 'SKIP'
]

pdf_with_tables = [fi['filename'] for fi in pdf_files if fi.get('has_tables')]
pdf_no_tables = [fi['filename'] for fi in pdf_files if not fi.get('has_tables')]

print(f'PDF files to process: {len(pdf_files)}')
print(f'  With tables: {len(pdf_with_tables)}')
print(f'  Without tables: {len(pdf_no_tables)}')

PDF files to process: 70
  With tables: 25
  Without tables: 45


## 2. Text Cleaning Utilities

In [3]:
# Common ligature replacements that PyMuPDF sometimes misses
LIGATURES = {
    '\ufb01': 'fi', '\ufb02': 'fl', '\ufb03': 'ffi',
    '\ufb04': 'ffl', '\ufb00': 'ff', '\ufb05': 'st',
    '\u2019': "'", '\u2018': "'", '\u201c': '"', '\u201d': '"',
    '\u2013': '-', '\u2014': '-', '\u00ad': '-',
    '\u00a0': ' ',
}

def clean_text(text: str) -> str:
    """Normalize extracted PDF text for downstream LLM processing."""
    # Replace ligatures
    for char, replacement in LIGATURES.items():
        text = text.replace(char, replacement)

    # Collapse excessive whitespace within lines
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        line = re.sub(r'[ \t]+', ' ', line).strip()
        cleaned_lines.append(line)

    # Collapse 3+ consecutive blank lines to 2
    text = '\n'.join(cleaned_lines)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()


def is_heading(span: dict, page_avg_fontsize: float) -> bool:
    """Heuristic: a text span is likely a heading if font is large or bold."""
    size = span.get('size', 0)
    flags = span.get('flags', 0)
    is_bold = bool(flags & 2**4)  # bit 4 = bold in PyMuPDF
    is_large = size > page_avg_fontsize * 1.15
    return is_bold or is_large


def extract_headings(page: fitz.Page) -> list[str]:
    """Extract likely section headings from a page using font size heuristics."""
    blocks = page.get_text('dict').get('blocks', [])
    all_sizes = [
        span['size']
        for b in blocks if b['type'] == 0
        for line in b['lines']
        for span in line['spans']
        if span.get('text', '').strip()
    ]
    avg_size = sum(all_sizes) / len(all_sizes) if all_sizes else 12.0

    headings = []
    for b in blocks:
        if b['type'] != 0:
            continue
        for line in b['lines']:
            line_text = ''.join(s['text'] for s in line['spans']).strip()
            if not line_text or len(line_text) > 120:
                continue
            if any(is_heading(s, avg_size) for s in line['spans']):
                headings.append(line_text)

    return headings


print('Text cleaning utilities defined')

Text cleaning utilities defined


## 3. Table Extraction - Markdown Conversion

In [4]:
def table_to_markdown(table) -> str:
    """
    Convert a PyMuPDF Table object to a markdown table string.
    Handles merged cells and empty values gracefully.
    """
    try:
        rows = table.extract() 
        if not rows:
            return ''

        # Normalize cells
        def norm(cell):
            if cell is None:
                return ''
            return re.sub(r'\s+', ' ', str(cell)).strip()

        norm_rows = [[norm(c) for c in row] for row in rows]
        # Filter fully empty rows
        norm_rows = [r for r in norm_rows if any(c for c in r)]
        if not norm_rows:
            return ''

        n_cols = max(len(r) for r in norm_rows)
        # Pad rows to same width
        norm_rows = [r + [''] * (n_cols - len(r)) for r in norm_rows]

        # Build markdown
        lines = []
        header = norm_rows[0]
        lines.append('| ' + ' | '.join(header) + ' |')
        lines.append('| ' + ' | '.join(['---'] * n_cols) + ' |')
        for row in norm_rows[1:]:
            lines.append('| ' + ' | '.join(row) + ' |')

        return '\n'.join(lines)

    except Exception as e:
        return f'[Table extraction error: {e}]'


print('Table-to-markdown converter defined')

Table-to-markdown converter defined


## 4. Main PDF Extractor

In [5]:
def extract_pdf(file_info: dict) -> dict:
    """
    Full extraction pipeline for a single PDF.
    Returns a structured dict ready to be saved as JSON.
    """
    abs_path = str(RAW_DIR / file_info['rel_path'])
    filename = file_info['filename']
    has_tables = file_info.get('has_tables', False)

    result = {
        'filename': filename,
        'source': file_info['rel_path'],
        'file_type': 'pdf',
        'has_tables': has_tables,
        'strategy': file_info.get('strategy', ''),
        'num_pages': 0,
        'pages': [],   # list of page dicts
        'all_headings': [],   # all detected headings across document
        'full_text': '',   # concatenated clean text (for agentic chunking)
        'tables': [],   # extracted tables as markdown
        'extracted_at': datetime.now().isoformat(),
        'error': None,
    }

    try:
        doc = fitz.open(abs_path)
        result['num_pages'] = doc.page_count

        full_text_parts = []
        all_headings = []
        all_tables = []

        for page_num, page in enumerate(doc):
            page_dict = {
                'page_num': page_num + 1,
                'text': '',
                'headings': [],
                'tables': [],
            }

            # Text extraction 
            raw_text = page.get_text('text')
            clean = clean_text(raw_text)
            page_dict['text'] = clean

            # Heading detection
            headings = extract_headings(page)
            page_dict['headings'] = headings
            all_headings.extend(headings)

            # Table extraction (only for flagged PDFs)
            if has_tables:
                tab_finder = page.find_tables()
                for tbl in tab_finder.tables:
                    md = table_to_markdown(tbl)
                    if md and len(md) > 20:  # skip trivially empty tables
                        table_entry = {
                            'page_num': page_num + 1,
                            'markdown': md,
                            'bbox': list(tbl.bbox),
                        }
                        page_dict['tables'].append(table_entry)
                        all_tables.append(table_entry)

            # Build full text with page separator
            page_header = f'\n\n--- Page {page_num + 1} ---\n'
            full_text_parts.append(page_header + clean)

            result['pages'].append(page_dict)

        doc.close()

        result['full_text'] = '\n'.join(full_text_parts).strip()
        result['all_headings'] = list(dict.fromkeys(all_headings))  # deduplicate, preserve order
        result['tables'] = all_tables

        # Append tables into full_text so agentic chunker sees them
        if all_tables:
            tables_section = '\n\n--- EXTRACTED TABLES ---\n'
            for i, tbl in enumerate(all_tables):
                tables_section += f'\nTable {i+1} (page {tbl["page_num"]}):\n{tbl["markdown"]}\n'
            result['full_text'] += tables_section

    except Exception as e:
        result['error'] = str(e)

    return result


print('PDF extractor defined')

PDF extractor defined


## 5. Run Extraction on All PDFs

In [6]:
print(f'Extracting {len(pdf_files)} PDF files...\n')

extraction_results = []
errors = []

for fi in tqdm(pdf_files, desc='Extracting PDFs'):
    extracted = extract_pdf(fi)

    if extracted['error']:
        errors.append({'file': fi['filename'], 'error': extracted['error']})
    else:
        extraction_results.append(extracted)

        # Save individual JSON per file
        safe_name = fi['filename'].replace('.pdf', '').replace(' ', '_').replace('/', '_')
        out_path = PDF_OUT_DIR / f'{safe_name}.json'
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(extracted, f, ensure_ascii=False, indent=2)

print(f'\nExtraction complete')
print(f'  Success : {len(extraction_results)}')
print(f'  Errors  : {len(errors)}')

if errors:
    print('\nErrors:')
    for e in errors:
        print(f'  - {e["file"]}: {e["error"]}')

Extracting 70 PDF files...



Extracting PDFs: 100%|██████████| 70/70 [01:51<00:00,  1.60s/it]


Extraction complete
  Success : 70
  Errors  : 0


## 6. Extraction Quality Check

In [7]:
print('EXTRACTION QUALITY REPORT')
print('=' * 70)
print(f'{"Filename":<50} {"Pages":>5} {"Words":>7} {"Tables":>7} {"Headings":>9}')
print('-' * 70)

total_words  = 0
total_tables = 0

for r in extraction_results:
    words = len(r['full_text'].split())
    n_tables = len(r['tables'])
    n_head = len(r['all_headings'])
    total_words += words
    total_tables += n_tables
    print(f'{r["filename"]:<50} {r["num_pages"]:>5} {words:>7,} {n_tables:>7} {n_head:>9}')

print('-' * 70)
print(f'{"TOTAL":<50} {"":>5} {total_words:>7,} {total_tables:>7}')

EXTRACTION QUALITY REPORT
Filename                                           Pages   Words  Tables  Headings
----------------------------------------------------------------------
AAS PTC 2022.pdf                                      47   8,572       2        50
Ad libitum dieting case study.pdf                      4     684       0         4
Ad libitum dieting PTC 2022.pdf                       73  16,169       0        53
Adherence PTC 2022.pdf                               107  31,507       1        87
Advanced strength training techniques PTC 2023.pdf    58  13,402       0        35
Age specific programming PTC 2022.pdf                 25   4,033       0        26
Biochemistry 101 PTC 2022.pdf                         14   1,396       0        12
Body fat percentage visual reference guide PTC.pdf    67   1,602       0        78
Carbohydrates PTC 2022.pdf                            91  19,573       4        54
Cardio PTC 2022.pdf                                   37   8,782       5 

## 7. Spot Check - Preview a Specific File

In [ ]:
# Change to any file you want to inspect
SPOT_CHECK_FILE = 'Case study average Joe program design.pdf'

r = next((x for x in extraction_results if x['filename'] == SPOT_CHECK_FILE), None)

if r:
    print(f'File: {r["filename"]}')
    print(f'Pages: {r["num_pages"]}')
    print(f'Tables: {len(r["tables"])}')
    print(f'Headings: {r["all_headings"][:10]}')
    print()
    print('--- Full text preview (first 1000 chars) ---')
    print(r['full_text'][:1000])
    print()
    if r['tables']:
        print('--- First extracted table ---')
        print(r['tables'][0]['markdown'][:600])
else:
    print(f'File not found in results: {SPOT_CHECK_FILE}')
    print(f'Available files: {[x["filename"] for x in extraction_results[:5]]}...')

File: Case study average Joe program design.pdf
Pages: 13
Tables: 5
Headings: ['CASE STUDY', 'AVERAGE JOE PROGRAM DESIGN', '\uf0b7 Full legal name: Ryan', '\uf0b7 Age: 23', '\uf0b7 Height: 176.5cm', '\uf0b7 Weight: 154lbs or 70kg', '\uf0b7 Body fat percentage (plus measurement technique): ~12%', '\uf0b7 Years of training experience: Too long! ~5 years.', '\uf0b7 Bench press: DB bench: 30kg each x6 .. Barbell doesn’t agree with my', 'shoulders. Best has been up to 36kg x6 (you can tell I have']

--- Full text preview (first 1000 chars) ---
--- Page 1 ---
BAYESIAN BODYBUILDING 2014

CASE STUDY
AVERAGE JOE PROGRAM DESIGN
Note: this is a copy of the anonymized client intake form from one of my clients with comments in
red and my client evaluation at the end.

The more information you provide, the better I will be able to assist you. If you have
any questions with regards to this form, please do not hesitate to contact me. You
can use whichever measurement system (e.g. metric, imperial) you

## 8. Spot Check - Intake Form (Critical for User Onboarding)

In [9]:
# The intake form is critical - it defines the structure of our user onboarding form
INTAKE_FILE = 'Henselmans Coaching Client Intake Form.docx'

# Note: docx is processed in notebook 03, but the PDF case studies contain
# filled-in intake forms. Let's preview one to understand the structure.
INTAKE_CASE = 'Case study average Joe program design.pdf'

r = next((x for x in extraction_results if x['filename'] == INTAKE_CASE), None)
if r:
    print('INTAKE FORM STRUCTURE (from case study):')
    print('This shows exactly what fields Henselmans uses for client intake.')
    print()
    print(r['full_text'][:2000])
else:
    print(f'{INTAKE_CASE} not found')

INTAKE FORM STRUCTURE (from case study):
This shows exactly what fields Henselmans uses for client intake.

--- Page 1 ---
BAYESIAN BODYBUILDING 2014

CASE STUDY
AVERAGE JOE PROGRAM DESIGN
Note: this is a copy of the anonymized client intake form from one of my clients with comments in
red and my client evaluation at the end.

The more information you provide, the better I will be able to assist you. If you have
any questions with regards to this form, please do not hesitate to contact me. You
can use whichever measurement system (e.g. metric, imperial) you prefer to fill in
this form. Just state which unit you used (e.g. cm, kg, lb, inch).

 Full legal name: Ryan
 Age: 23
 Height: 176.5cm
 Weight: 154lbs or 70kg
 Body fat percentage (plus measurement technique): ~12%
 Years of training experience: Too long! ~5 years.
For the section below, please give your current maximum abilities in weight times
reps, e.g. 250 lb x 6.
 Bench press: DB bench: 30kg each x6 .. Barbell doesn't ag

## 9. Save Extraction Summary

In [11]:
summary = {
    'extracted_at': datetime.now().isoformat(),
    'total_pdfs': len(pdf_files),
    'successful': len(extraction_results),
    'errors': errors,
    'total_words': total_words,
    'total_tables_found': total_tables,
    'files': [
        {
            'filename': r['filename'],
            'source': r['source'],
            'num_pages': r['num_pages'],
            'word_count': len(r['full_text'].split()),
            'table_count': len(r['tables']),
            'heading_count': len(r['all_headings']),
            'strategy': r['strategy'],
            'output_file': r['filename'].replace('.pdf', '').replace(' ', '_') + '.json',
        }
        for r in extraction_results
    ]
}

summary_path = PROCESSED_DIR / 'pdf_extraction_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f'Summary saved -> {summary_path}')
print()
print('=' * 60)
print('  NOTEBOOK 02 — COMPLETE')
print('=' * 60)
print(f'  PDFs extracted: {len(extraction_results)} / {len(pdf_files)}')
print(f'  Total words: {total_words:,}')
print(f'  Tables extracted: {total_tables}')
print(f'  Output dir: data/processed/pdf_extracted/')

Summary saved -> D:\Cybernetic Gym Assistant\backend\data\processed\pdf_extraction_summary.json

  NOTEBOOK 02 — COMPLETE
  PDFs extracted: 70 / 70
  Total words: 541,200
  Tables extracted: 211
  Output dir: data/processed/pdf_extracted/
